# Patient Health Data Preprocessing & Feature Engineering
**Educational project | Synthetic data | 5,000 records**

This notebook follows the supplied assignment: Part A (missing values), Part B (outliers), and Part C (final clean dataset). It does not contain real patient records and is not for clinical diagnosis.

## 1. Import libraries and load dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df = pd.read_csv('../Dataset/raw_patient_data.csv')
print('Shape:', df.shape)
display(df.head())
display(df.info())

ModuleNotFoundError: No module named 'sklearn'

## 2. Dataset overview and missing-value summary

In [ ]:
numeric_cols = ['age', 'bmi', 'blood_pressure', 'cholesterol', 'glucose']
categorical_cols = ['gender', 'region']
outlier_cols = ['bmi', 'blood_pressure', 'cholesterol', 'glucose']

missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100).round(2)
})
display(missing_summary)
missing_summary.to_csv('../Reports/Missing_Values_Report.csv')

NameError: name 'df' is not defined

## Part A — Handling Missing Values
The target variable `disease_risk` is not imputed or modified. Numeric and categorical columns are treated separately.

### A1. Simple Imputer — median for numeric, most frequent for categorical

In [ ]:
simple_df = df.copy()
simple_df[numeric_cols] = SimpleImputer(strategy='median').fit_transform(simple_df[numeric_cols])
simple_df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(simple_df[categorical_cols])
print('Remaining missing values:', simple_df.isna().sum().sum())

### A2. Most Frequent Imputation
For categorical columns, replace missing values with the most frequently occurring category.

In [ ]:
most_frequent_df = df.copy()
most_frequent_df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(most_frequent_df[categorical_cols])
display(most_frequent_df[categorical_cols].isna().sum())

### A3. Missing Indicator + Random Sample Imputation
Create a binary flag for missingness, then sample replacements from observed values in the same column.

In [ ]:
random_df = df.copy()
rng_local = np.random.default_rng(42)
for col in numeric_cols + categorical_cols:
    missing_mask = random_df[col].isna()
    random_df[col + '_was_missing'] = missing_mask.astype(int)
    observed = random_df.loc[~missing_mask, col].dropna().to_numpy()
    if missing_mask.any():
        random_df.loc[missing_mask, col] = rng_local.choice(observed, missing_mask.sum(), replace=True)
print('Remaining missing values:', random_df[numeric_cols + categorical_cols].isna().sum().sum())

### A4. KNN Imputer
KNN estimates missing numeric values using nearby rows. Categorical columns use most-frequent imputation because standard KNNImputer expects numeric inputs.

In [ ]:
knn_df = df.copy()
knn_df[numeric_cols] = KNNImputer(n_neighbors=5, weights='distance').fit_transform(knn_df[numeric_cols])
knn_df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(knn_df[categorical_cols])
print('Remaining missing values:', knn_df.isna().sum().sum())

### A5. MICE / Iterative Imputer
IterativeImputer models each numeric feature from the other numeric features in repeated rounds. It is used here as a practical MICE-style implementation.

In [ ]:
mice_df = df.copy()
mice_imputer = IterativeImputer(random_state=42, max_iter=10)
mice_df[numeric_cols] = mice_imputer.fit_transform(mice_df[numeric_cols])
mice_df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(mice_df[categorical_cols])
print('Remaining missing values:', mice_df.isna().sum().sum())
mice_df.to_csv('../Dataset/mice_imputed_intermediate.csv', index=False)

### A6. Compare imputation approaches

In [ ]:
imputation_comparison = pd.DataFrame({
    'method': ['Simple (median + most frequent)', 'Most frequent (categorical only)',
               'Missing indicator + random sample', 'KNN (numeric) + most frequent', 'MICE-style iterative (numeric) + most frequent'],
    'remaining_missing_values': [
        int(simple_df.isna().sum().sum()),
        int(most_frequent_df[categorical_cols].isna().sum().sum()),
        int(random_df[numeric_cols + categorical_cols].isna().sum().sum()),
        int(knn_df.isna().sum().sum()),
        int(mice_df.isna().sum().sum())
    ]
})
display(imputation_comparison)
imputation_comparison.to_csv('../Reports/Imputation_Method_Comparison.csv', index=False)

## Part B — Handling Outliers
Detection methods: Z-score, IQR, and percentile. Treatment: Winsorization (cap at the 1st and 99th percentiles).

In [ ]:
outlier_rows = []
for col in outlier_cols:
    s = mice_df[col].dropna()
    z_score = (s - s.mean()) / s.std(ddof=0)
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    p01, p99 = s.quantile(.01), s.quantile(.99)
    outlier_rows.append({
        'column': col,
        'z_score_count_abs_gt_3': int((z_score.abs() > 3).sum()),
        'iqr_count_1_5_rule': int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()),
        'percentile_count_1_99': int(((s < p01) | (s > p99)).sum()),
        'p01_lower_bound': p01, 'p99_upper_bound': p99
    })
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)
outlier_report.to_csv('../Reports/Outlier_Detection_Report.csv', index=False)

### B1. Visualize outliers before treatment

In [ ]:
mice_df[outlier_cols].boxplot(figsize=(9, 4))
plt.title('Outlier Check Before Treatment')
plt.ylabel('Value')
plt.show()

### B2. Winsorization

In [ ]:
final_df = mice_df.copy()
winsor_rows = []
for col in outlier_cols:
    lower = final_df[col].quantile(.01)
    upper = final_df[col].quantile(.99)
    below = int((final_df[col] < lower).sum())
    above = int((final_df[col] > upper).sum())
    final_df[col] = final_df[col].clip(lower=lower, upper=upper)
    winsor_rows.append([col, lower, upper, below, above])
winsor_report = pd.DataFrame(winsor_rows, columns=['column','lower_cap_p01','upper_cap_p99','values_below_cap','values_above_cap'])
display(winsor_report)
winsor_report.to_csv('../Reports/Winsorization_Report.csv', index=False)

## Part C — Final Clean Dataset

In [ ]:
# Verify the target variable remains unchanged
assert final_df['disease_risk'].equals(df['disease_risk'])
assert final_df.isna().sum().sum() == 0

quality_report = pd.DataFrame([
    ['Raw dataset', len(df), int(df.isna().sum().sum()), int(df.duplicated().sum())],
    ['Final cleaned dataset', len(final_df), int(final_df.isna().sum().sum()), int(final_df.duplicated().sum())]
], columns=['dataset','rows','missing_values','duplicate_rows'])
display(quality_report)
quality_report.to_csv('../Reports/Before_After_Quality_Report.csv', index=False)
final_df.to_csv('../Dataset/cleaned_patient_data.csv', index=False)
display(final_df.head())
print('Final dataset saved.')

## 4. Final visualizations

In [ ]:
final_df[outlier_cols].boxplot(figsize=(9, 4))
plt.title('Clinical Measurements After Winsorization')
plt.ylabel('Value')
plt.show()

final_df['disease_risk'].value_counts().sort_index().plot(kind='bar', figsize=(5, 3))
plt.title('Synthetic Disease Risk Class Distribution')
plt.xlabel('disease_risk (0=Low, 1=High)')
plt.ylabel('Patient count')
plt.show()

## 5. Brief report / conclusion

- MICE-style IterativeImputer was selected for numeric missing values because it models each feature using the other numeric features.
- Most-frequent imputation was used for categorical values.
- Z-score, IQR, and percentile methods were used to identify potential outliers.
- Winsorization capped extreme numeric measurements at the 1st and 99th percentiles rather than deleting rows.
- The final dataset contains no missing values, and `disease_risk` was preserved.
- All records are synthetic and intended only for learning preprocessing techniques.